# Initiation à l'API Berserk

L'objectif de ce notebook est de comprendre comment utiliser l'API Berserk.

## 1. Imports de libraries

In [1]:
from dotenv import load_dotenv

import os
import sys
sys.path.append(os.path.abspath(".."))

from pprint import pprint

import berserk
import pandas as pd

## 2. Chargement du token

In [2]:
load_dotenv("../.env")

token = os.getenv("LICHESS_API_TOKEN")

if not token:
    raise ValueError("LICHESS_API_TOKEN introuvable dans le fichier .env")

print("Token chargé avec succès")

Token chargé avec succès


## 3. Connexion à l'API Berserk

In [3]:
session = berserk.TokenSession(token)
client = berserk.Client(session=session)

print("Client Berserk créé avec succès")

Client Berserk créé avec succès


## 4. Affichage des informations de mon profil Lichess

In [4]:
account = client.account.get()
pprint(account)

{'blocking': False,
 'count': {'all': 1,
           'bookmark': 0,
           'draw': 0,
           'import': 0,
           'loss': 0,
           'me': 0,
           'playing': 0,
           'rated': 1,
           'win': 1},
 'createdAt': datetime.datetime(2026, 3, 17, 18, 3, 37, 330000, tzinfo=datetime.timezone.utc),
 'followable': True,
 'following': False,
 'id': 'raydennnnnnnnn',
 'perfs': {'blitz': {'games': 1,
                     'prog': 0,
                     'prov': True,
                     'rating': 1674,
                     'rd': 296},
           'bullet': {'games': 0,
                      'prog': 0,
                      'prov': True,
                      'rating': 1500,
                      'rd': 500},
           'classical': {'games': 0,
                         'prog': 0,
                         'prov': True,
                         'rating': 1500,
                         'rd': 500},
           'correspondence': {'games': 0,
                              'prog'

## 5. Importation des parties d'un joueur

Dans cet exemple, il est importé les parties du joueur Chesstam. Un maximim de 5 parties sont importées. Seules des parties jouées en mode rapide sont considérées.

In [5]:
username = "Chesstam"

games = client.games.export_by_player(
    username,
    max=5,
    perf_type="rapid",
    opening=True,
)

games = list(games)

print(f"{len(games)} parties récupérées")

5 parties récupérées


## 6. Inspection d'une partie

Cette première commande permet d'extraire toutes les informations d'une partie. L'API a converti le format PGN en JSON.

In [6]:
pprint(games[0])

{'clock': {'increment': 0, 'initial': 600, 'totalTime': 600},
 'createdAt': datetime.datetime(2026, 3, 17, 19, 32, 27, 908000, tzinfo=datetime.timezone.utc),
 'id': 'Y2Mfy0dv',
 'lastMoveAt': datetime.datetime(2026, 3, 17, 19, 45, 51, 29000, tzinfo=datetime.timezone.utc),
 'moves': 'e3 d5 d4 Nf6 f3 Bf5 Ne2 h6 Ng3 Bh7 Bd3 e6 Bxh7 Nxh7 O-O Bd6 f4 g5 '
          'fxg5 Bxg3 hxg3 Qxg5 e4 Qxg3 Qf3 Qxf3 Rxf3 dxe4 Re3 Nd7 Rxe4 O-O-O '
          'c3 b6 Nd2 Ng5 Re3 f6 Ne4 e5 dxe5 Nxe5 Nxg5 hxg5 Re2 Rd1+ Kf2 Ng4+ '
          'Kg3 Rh4 Rd2 Re1 Rd3 Ne5 Re3 Rd1 a4 Nc4 Rf3 Rd6 Be3 Ne5 Rf5 Rd3 Re1 '
          'Nc4 Kf2 Nxb2 Rxf6 Nxa4 Bxg5 Rc4 Rf8+ Kb7 Rf7 Rcxc3 Bf6 Rc2+ Kg1 Ka6 '
          'Bh4 c5 Rf4 Nc3 Ra1+ Kb5 Rxa7 Rd1+ Kh2 Ne2 Rf2 Rd4 g3 Rcd2 Re7 Nc3 '
          'Rxd2 Rxd2+ Kh3 Rd3 Bf6 Nd5 Re6 Nf4+ Kg4 Nxe6 Bh8 c4 Kh4 c3 g4 c2 '
          'Bb2 Rb3 Bc1 Rb1 Be3 c1=Q Bxc1 Rxc1 g5 Rc7 Kh5 Kc6 g6 b5 Kg4 b4 Kf5 '
          'Ng7+ Kf6 b3',
 'opening': {'eco': 'D00', 'name': "Queen's Pawn Game", 'ply': 4},


Cette seconde commande permet d'extraire les différents champs d'une partie.

In [7]:
print(sorted(games[0].keys()))

for key in sorted(games[0].keys()):
    print(key)

['clock', 'createdAt', 'id', 'lastMoveAt', 'moves', 'opening', 'perf', 'players', 'rated', 'source', 'speed', 'status', 'variant', 'winner']
clock
createdAt
id
lastMoveAt
moves
opening
perf
players
rated
source
speed
status
variant
winner


Les prochaines commandes permettent d'afficher les sous-champs des champs "players" et "opening".

In [8]:
for key in sorted(games[0]["players"].keys()):
    print(key)

print("\n")

for key in sorted(games[0]["players"]["white"].keys()):
    print(key)

print("\n")

for key in sorted(games[0]["players"]["white"]["user"].keys()):
    print(key)

print("\n")

black
white


rating
ratingDiff
user


flair
id
name




In [9]:
for key in sorted(games[0]["opening"].keys()):
    print(key)

eco
name
ply


# 7. Applatissement d'une partie

La commande suivante éxécute une fonction permettant de convertir una partie brute en une structure plate.

In [10]:
from src.ingestion.flatten_games import flatten_game

flat = flatten_game(games[0])
flat

{'game_id': 'Y2Mfy0dv',
 'datetime_utc': datetime.datetime(2026, 3, 17, 19, 32, 27, 908000, tzinfo=datetime.timezone.utc),
 'perf': 'rapid',
 'speed': 'rapid',
 'rated': True,
 'source': 'pool',
 'status': 'resign',
 'winner': 'black',
 'white_id': 'felixpito',
 'white_name': 'Felixpito',
 'black_id': 'chesstam',
 'black_name': 'Chesstam',
 'white_rating_before': 1476,
 'black_rating_before': 1376,
 'white_rating_diff': -7,
 'black_rating_diff': 7,
 'white_rating_after': 1469,
 'black_rating_after': 1383,
 'opening_eco': 'D00',
 'opening_name': "Queen's Pawn Game",
 'clock_initial': 600,
 'clock_increment': 0,
 'has_increment': 0,
 'moves': 'e3 d5 d4 Nf6 f3 Bf5 Ne2 h6 Ng3 Bh7 Bd3 e6 Bxh7 Nxh7 O-O Bd6 f4 g5 fxg5 Bxg3 hxg3 Qxg5 e4 Qxg3 Qf3 Qxf3 Rxf3 dxe4 Re3 Nd7 Rxe4 O-O-O c3 b6 Nd2 Ng5 Re3 f6 Ne4 e5 dxe5 Nxe5 Nxg5 hxg5 Re2 Rd1+ Kf2 Ng4+ Kg3 Rh4 Rd2 Re1 Rd3 Ne5 Re3 Rd1 a4 Nc4 Rf3 Rd6 Be3 Ne5 Rf5 Rd3 Re1 Nc4 Kf2 Nxb2 Rxf6 Nxa4 Bxg5 Rc4 Rf8+ Kb7 Rf7 Rcxc3 Bf6 Rc2+ Kg1 Ka6 Bh4 c5 Rf4 Nc3 Ra

# 8. Conversion en une partie du point de vue du joueur

La commande suivante fait appel à une fonction permettant de transformer une partie aplatie en 2 rangées (1 par joueur).

In [11]:
from src.ingestion.player_view import game_to_player_rows

flat = flatten_game(games[1])
rows = game_to_player_rows(flat)

rows

[{'game_id': 'tMUQMqEi',
  'datetime_utc': datetime.datetime(2026, 3, 17, 19, 17, 8, 521000, tzinfo=datetime.timezone.utc),
  'weekday': 1,
  'player_id': 'chesstam',
  'player_name': 'Chesstam',
  'opponent_id': 'juleskeats',
  'opponent_name': 'juleskeats',
  'color_player': 'white',
  'result_player': 0.0,
  'elo_player_before': 1382,
  'elo_player_after': 1376,
  'elo_diff_player': -6,
  'termination_type': 'mate',
  'has_increment': 0,
  'opening_family': 'Rapport-Jobava System',
  'opening_eco': 'D01',
  'opening_name': 'Rapport-Jobava System',
  'ply_count': 46,
  'perf': 'rapid',
  'speed': 'rapid',
  'rated': True,
  'source': 'pool'},
 {'game_id': 'tMUQMqEi',
  'datetime_utc': datetime.datetime(2026, 3, 17, 19, 17, 8, 521000, tzinfo=datetime.timezone.utc),
  'weekday': 1,
  'player_id': 'juleskeats',
  'player_name': 'juleskeats',
  'opponent_id': 'chesstam',
  'opponent_name': 'Chesstam',
  'color_player': 'black',
  'result_player': 1.0,
  'elo_player_before': 1333,
  'elo_

# 9. Table complète des parties du joueur

La commande suivante permet de convertir un ensemble de parties brutes en un ensemble de parties aplaties en 2 rangées (format de la section précédente).

In [20]:
from src.ingestion.build_player_games import build_player_games

df_player_games = build_player_games(games)

print(len(games))

print(len(df_player_games))

print(df_player_games.columns)

df_player_games.head()



5
10
Index(['game_id', 'datetime_utc', 'weekday', 'player_id', 'player_name',
       'opponent_id', 'opponent_name', 'color_player', 'result_player',
       'elo_player_before', 'elo_player_after', 'elo_diff_player',
       'termination_type', 'has_increment', 'opening_family', 'opening_eco',
       'opening_name', 'ply_count', 'perf', 'speed', 'rated', 'source'],
      dtype='str')


,game_id,datetime_utc,weekday,player_id,player_name,opponent_id,opponent_name,color_player,result_player,elo_player_before,...,termination_type,has_increment,opening_family,opening_eco,opening_name,ply_count,perf,speed,rated,source
0,Y2Mfy0dv,2026-03-17 19:32:27.908000+00:00,1,felixpito,Felixpito,chesstam,Chesstam,white,0.0,1476,...,resign,0,Queen's Pawn Game,D00,Queen's Pawn Game,132,rapid,rapid,True,pool
1,Y2Mfy0dv,2026-03-17 19:32:27.908000+00:00,1,chesstam,Chesstam,felixpito,Felixpito,black,1.0,1376,...,resign,0,Queen's Pawn Game,D00,Queen's Pawn Game,132,rapid,rapid,True,pool
2,tMUQMqEi,2026-03-17 19:17:08.521000+00:00,1,chesstam,Chesstam,juleskeats,juleskeats,white,0.0,1382,...,mate,0,Rapport-Jobava System,D01,Rapport-Jobava System,46,rapid,rapid,True,pool
3,tMUQMqEi,2026-03-17 19:17:08.521000+00:00,1,juleskeats,juleskeats,chesstam,Chesstam,black,1.0,1333,...,mate,0,Rapport-Jobava System,D01,Rapport-Jobava System,46,rapid,rapid,True,pool
4,G2NX03pO,2026-03-17 19:07:26.370000+00:00,1,soullforged,soullforged,chesstam,Chesstam,white,0.0,1377,...,mate,0,Scandinavian Defense,B01,Scandinavian Defense: Mieses-Kotroc Variation,56,rapid,rapid,True,pool
